In [ ]:
from scripts.utils import *
from scripts.evcouplings import *
from scripts.get_structures_and_sequences import *
from scripts.structural_filter import *
from scripts.hmm_domain_filter import *

In [4]:
# IMPORTANT PARAMETERS TO SET BEFORE RUNNING:
# evcouplings, foldseek, USalign, hmmpress and hmmsearch should be installed and in Path

SEED_uid = "Q5SK37"
SEED_region_start = 507
SEED_region_end = 569
SEED = [SEED_uid, SEED_region_start, SEED_region_end]
BIT_SCORES = [0.8]
TM_SCORE_CUTOFF = 0.6
HMMs_DIR = "STAC_hmm_domains"
CPUS = 16

CONFIG_TEMPLATE = "evcouplings_config_template.yaml"
number_tracker = {} # dictionary to keep track of true members at each stage in the pipeline

In [ ]:


def setup_directories():
    root_dir = f"runs/true_members_run_{SEED[0]}_{time.strftime('%Y-%m-%d_%H-%M-%S')}"
    directories = [
        f"{root_dir}/data/evcouplings",
        f"{root_dir}/data/filtering/usalign",
        f"{root_dir}/data/filtering/foldseek",
        f"{root_dir}/data/filtering/hmm_domain_filtering",
        f"{root_dir}/data/alphafold_structures",
        f"{root_dir}/data/seed",
        f"{root_dir}/data/HMMs",
        f"{root_dir}/data/results"
    ]
    for directory in directories:
        # clear directory if it already exists
        if os.path.exists(directory):
            shutil.rmtree(directory)
        os.makedirs(directory, exist_ok=True)
    return root_dir
        
#root_dir = setup_directories()
#print(root_dir)
root_dir = "runs/true_members_run_Q5SK37_2026-06-11_09-56-39"

In [6]:
# Run evcouplings align and couplings stage
#run_evcouplings(SEED, BIT_SCORES, CPUS, CONFIG_TEMPLATE, root_dir)

In [7]:
# Run foldseek webserver search and filter output with TM_SCORE_CUTOFF

# get correct roi of seed
seed_uid = SEED[0]
seed_r_start = SEED[1]
seed_r_end = SEED[2]
run_download_tasks([seed_uid], output_dir=f"{root_dir}/data/seed/", max_workers=1)
query_structure_full = f"{root_dir}/data/seed/{seed_uid}.cif"
query_structure_roi = extract_region_of_protein(query_structure_full, (seed_r_start, seed_r_end))

number_tracker = run_foldseek_webserver(number_tracker, query_structure_roi, root_dir, TM_SCORE_CUTOFF)

Processing 1/1 structures...
Downloaded 1/1 structure files (0 failed).
Extracting region (507, 569) from protein file: runs/true_members_run_Q5SK37_2026-06-11_09-56-39/data/seed/Q5SK37.cif


TypeError: run_foldseek_webserver() missing 1 required positional argument: 'out_dir'

In [ ]:
# Get all structures and full fasta of all sequences of evcouplings output
number_tracker = get_all_structures_and_sequences(SEED, BIT_SCORES, number_tracker, root_dir)

Could not extract UniProt accession from record ID: UniRef90_UPI00263A1D77/508-565
Could not extract UniProt accession from record ID: UniRef90_UPI001FAAB0B1/508-565
Could not extract UniProt accession from record ID: UniRef90_UPI00023898AB/505-563
Could not extract UniProt accession from record ID: UniRef90_UPI000DF2BD3B/508-565
Could not extract UniProt accession from record ID: UniRef90_UPI0003824F5A/507-565
Could not extract UniProt accession from record ID: UniRef90_UPI000542879D/506-563
            id  region_start  region_end
0   A0A0G1N724             6          68
1   A0A0M9AE96           506         563
2   A0A0N0ZNZ1           507         565
3   A0A1F5SF33             6          65
4   A0A1F5SFS3             7          66
5   A0A1F5Z9A6             7          65
6   A0A1J0LTU8           504         561
7   A0A2G9C2J8            15          77
8   A0A2M6P0Q7            11          69
9   A0A2M7Z5X0            10          68
10  A0A3P4APY8           507         569
11  A0A430

In [ ]:
# Run USalgin, foldseek and filter output
number_tracker = run_structural_similarity_filter(SEED, number_tracker, TM_SCORE_CUTOFF, root_dir)

Remaining structures after filtering of foldseek output: 29/32
Remaining structures after filtering usalign output: 30/54


In [ ]:
# Run hmmsearch with CbrA related HMMs on fasta of STAC containing sequences and filter to get STAC containing CbrAs and STAC containing proteins without CbrA domains
seed_containing_sequences_ids_path = f"{root_dir}/data/filtering/structural_filtering_output_tm_0.6.csv"
number_tracker = run_hmm_domain_filter(number_tracker, CPUS, seed_containing_sequences_ids_path, HMMs_DIR, root_dir)

ID 'A0A7C5VEJ4' not found in the FASTA file: runs/true_members_run_Q5SK37_2026-06-11_09-56-39/data/all_sequences_full.fasta
ID 'A0A430RH80' not found in the FASTA file: runs/true_members_run_Q5SK37_2026-06-11_09-56-39/data/all_sequences_full.fasta
ID 'A0A0N0ZNZ1' not found in the FASTA file: runs/true_members_run_Q5SK37_2026-06-11_09-56-39/data/all_sequences_full.fasta
ID 'A0A8T6Y4X3' not found in the FASTA file: runs/true_members_run_Q5SK37_2026-06-11_09-56-39/data/all_sequences_full.fasta
ID 'A0A8S5UAS9' not found in the FASTA file: runs/true_members_run_Q5SK37_2026-06-11_09-56-39/data/all_sequences_full.fasta
ID 'A0A5C6U3H4' not found in the FASTA file: runs/true_members_run_Q5SK37_2026-06-11_09-56-39/data/all_sequences_full.fasta
ID 'A0A2M7Z5X0' not found in the FASTA file: runs/true_members_run_Q5SK37_2026-06-11_09-56-39/data/all_sequences_full.fasta
ID 'A0A1F5Z9A6' not found in the FASTA file: runs/true_members_run_Q5SK37_2026-06-11_09-56-39/data/all_sequences_full.fasta
ID 'A0A5

RuntimeError: hmmsearch failed with error:

Error: Sequence file runs/true_members_run_Q5SK37_2026-06-11_09-56-39/data/filtering/hmm_domain_filtering/seed_containing_sequences.fasta is empty or misformatted



In [ ]:
# Print number of true members at each stage in the pipeline and save results in results directory
for key, value in number_tracker.items():
    print(f"{key}: {value}")
    
df_number_tracker = pd.DataFrame(list(number_tracker.items()), columns=["Stage", "Numbers"])
df_number_tracker.to_csv(f"{root_dir}/data/results/number_tracker.csv", index=False)

copy_file(f"{root_dir}/data/filtering/True_Members.csv", f"{root_dir}/data/results/True_Members.csv")
copy_file(f"{root_dir}/data/filtering/structures_with_seed_and_not_all_domains.csv", f"{root_dir}/data/results/structures_with_seed_and_not_all_domains.csv")
copy_file(f"{root_dir}/data/filtering/hmm_domain_filtering/hmmsearch_results.domtblout", f"{root_dir}/data/results/hmmsearch_results.domtblout")

number_filtered_structures_foldseek_webserver: 48
annotation_files_jackhmmer: 1
unique_ids_jackhmmer: 13
unique_ids_combined: 56
unique_uniprot_uids: 56
successful_structure_downloads: 54
downloaded_records: 30
number_filtered_structures_foldseek: 29
number_filtered_structures_usalign: 30
number_filtered_structures_total: 30
structures_with_seed_and_domains: 0
structures_with_seed_and_not_all_domains: 30


True

In [ ]:
# insert here the analysis of the results